In [1]:
"""
MAP - Charting Student Math Misunderstandings - Inference Notebook v3
改用 log-likelihood ranking 取代 beam search:
對每個 test 樣本,計算模型生成所有已知 label 的機率,排序取 top-3。
"""
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

# ==== 路徑 ====
BASE_MODEL_PATH = "/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1"
ADAPTER_PATH = "/kaggle/input/datasets/alextsai2004/gemma-math-misunderstanding-lora/best_gemma_lora_model"
TEST_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/test.csv"
TRAIN_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv"
SAMPLE_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/sample_submission.csv"
OUTPUT_CSV = "/kaggle/working/submission.csv"

# 同 prompt 下一次掃多少個 label;OOM 就降,T4 16GB 上 32 蠻安全
LABEL_BATCH_SIZE = 32


# ==== Sample submission 當格式骨架 ====
print("=" * 60)
sample = pd.read_csv(SAMPLE_CSV)
print(f"Sample shape: {sample.shape}")
print(f"Columns: {sample.columns.tolist()}")
ROW_ID_COL = sample.columns[0]
PRED_COL = sample.columns[1]


# ==== Prompt(完全照訓練時) ====
def build_user_prompt(question, correct_answer, student_explanation):
    return (
        "You are a math misconception classifier.\n"
        "Given the question, the correct answer, and the student's explanation, "
        "predict the final label in the format `Category:Misconception`.\n"
        "If the category is not a misconception type, use `NA` for the misconception part.\n\n"
        f"Question: {question}\n"
        f"Correct answer: {correct_answer}\n"
        f"Student explanation: {student_explanation}\n\n"
        "Return only the label."
    )


def build_prompt(row):
    user = build_user_prompt(
        row["QuestionText"], row["MC_Answer"], row["StudentExplanation"]
    )
    return f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n"


# ==== Model ====
print("Loading base model in bf16 ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
device = next(model.parameters()).device
pad_id = tokenizer.pad_token_id


# ==== 從訓練集列出所有出現過的 label,並 tokenize ====
train_df = pd.read_csv(TRAIN_CSV)
train_df["target"] = (
    train_df["Category"].astype(str) + ":" +
    train_df["Misconception"].fillna("NA").astype(str)
)
unique_labels = sorted(train_df["target"].unique().tolist())
print(f"Number of unique labels: {len(unique_labels)}")

# 每個 label 後面接 <end_of_turn>,跟訓練時模型學到的輸出格式一致
label_token_ids = [
    tokenizer.encode(label + "<end_of_turn>", add_special_tokens=False)
    for label in unique_labels
]
label_lengths = [len(ids) for ids in label_token_ids]
print(f"Label token length: min={min(label_lengths)}, "
      f"max={max(label_lengths)}, mean={np.mean(label_lengths):.1f}")


# ==== Test ====
test_df = pd.read_csv(TEST_CSV)
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    test_df[col] = test_df[col].fillna("")
print(f"Test size: {len(test_df)}")
assert len(test_df) == len(sample), \
    f"test {len(test_df)} != sample {len(sample)}"


# ==== Log-likelihood scoring ====
@torch.no_grad()
def score_all_labels(prompt_text):
    """
    回傳 array of shape [num_labels],每個值是該 label 的長度正規化 log-likelihood。
    """
    # Prompt 只 tokenize 一次
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=True)

    scores = np.empty(len(label_token_ids), dtype=np.float32)

    for start in range(0, len(label_token_ids), LABEL_BATCH_SIZE):
        end = min(start + LABEL_BATCH_SIZE, len(label_token_ids))
        batch_labels = label_token_ids[start:end]
        B = len(batch_labels)

        # 組 sequences: prompt + label,左 padding 對齊到最右邊
        sequences = [prompt_ids + lbl for lbl in batch_labels]
        max_len = max(len(s) for s in sequences)

        input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((B, max_len), dtype=torch.long)
        label_starts = []
        for j, seq in enumerate(sequences):
            pad = max_len - len(seq)
            input_ids[j, pad:] = torch.tensor(seq, dtype=torch.long)
            attention_mask[j, pad:] = 1
            label_starts.append(max_len - len(batch_labels[j]))

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # 一次 forward
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        # logits: [B, max_len, vocab]

        # 逐筆算 log-prob
        for j, lbl in enumerate(batch_labels):
            L = len(lbl)
            ls = label_starts[j]
            # 預測 label[k] 用的是位置 ls-1+k 的 logits
            slice_logits = logits[j, ls - 1:ls - 1 + L, :].float()
            log_probs = torch.log_softmax(slice_logits, dim=-1)
            target = torch.tensor(lbl, device=device)
            tok_lp = log_probs.gather(1, target.unsqueeze(1)).squeeze(1)
            scores[start + j] = tok_lp.mean().item()  # length-normalized

    return scores


# ==== 主迴圈 ====
unique_labels_arr = np.array(unique_labels)
pred_dict = {}

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Scoring"):
    prompt = build_prompt(row)
    scores = score_all_labels(prompt)
    # 取分數最高的 3 個 label
    top3_idx = np.argpartition(-scores, 3)[:3]
    # argpartition 不保證內部排序,再 sort 一次
    top3_idx = top3_idx[np.argsort(-scores[top3_idx])]
    top3 = unique_labels_arr[top3_idx].tolist()
    pred_dict[row["row_id"]] = " ".join(top3)


# ==== 用 sample 當骨架建 submission ====
submission = sample.copy()
submission[PRED_COL] = submission[ROW_ID_COL].map(pred_dict)

print("\n" + "=" * 60)
print("Validation:")
print(f"Shape: {submission.shape}")
print(f"Any NaN: {submission.isna().any().any()}")
print(f"Head:\n{submission.head()}")

assert submission.shape == sample.shape
assert not submission.isna().any().any()
assert (submission[PRED_COL] != "").all()
assert (submission[PRED_COL].str.split().str.len() == 3).all()

submission.to_csv(OUTPUT_CSV, index=False)
print(f"\n[OK] Saved {OUTPUT_CSV}")

Sample shape: (3, 2)
Columns: ['row_id', 'Category:Misconception']
Loading base model in bf16 ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Number of unique labels: 65
Label token length: min=6, max=14, mean=9.6
Test size: 3


Scoring: 100%|██████████| 3/3 [00:26<00:00,  8.77s/it]


Validation:
Shape: (3, 2)
Any NaN: False
Head:
   row_id                             Category:Misconception
0   36696  True_Neither:NA True_Correct:NA True_Misconcep...
1   36697  False_Neither:NA False_Misconception:WNB False...
2   36698  True_Neither:NA True_Correct:NA True_Misconcep...

[OK] Saved /kaggle/working/submission.csv
